# Programming Assignment: Movie Recommendation with Learned Embeddings

- **과제 목표**: 같은 사용자가 본 영화 쌍 $(m_1, m_2)$ 으로 뉴럴 네트워크를 학습해 **영화 임베딩**을 만들고, 이 임베딩으로 각 사용자에게 맞춤형 영화를 추천한다.
- 강의자료 *04_Model-based Recommender Systems_part2* **slide 57–61** 의 "Example: Recommending Movies" 를 그대로 구현하는 과제입니다.

| 단계 | 슬라이드 | 이 과제에서 하는 일 |
|---|---|---|
| Train/Test split | 58 | 사용자마다 **가장 최근에 본 5편**을 test 로 떼어냄 |
| Train data 생성 | 59 | 같은 사용자가 본 영화들로 $(m_1, m_2)$ pair 생성 |
| 신경망 학습 | 60 | one-hot($m_1$) → embedding → softmax 로 $m_2$ 예측 (Cross-entropy) |
| 추천 | 61 | 임베딩 유사도가 높은 영화 Top-10 추천 |

# Notice (꼭 읽어주세요)

**채점 방식**: 제출한 노트북을 조교가 **처음부터 다시 실행**한 뒤, 아래 함수들을 **숨겨진 테스트 데이터**로 자동 테스트합니다.
따라서 다음 규칙을 반드시 지켜주세요. 규칙을 어겨 자동 채점이 실패하면 해당 문항은 0점 처리될 수 있습니다.

1. **`# YOUR CODE HERE` 부분만 작성**하고, `raise NotImplementedError()` 줄은 지워주세요.
2. **함수 이름, 인자, 반환 형식, 클래스/속성 이름(`W_in`, `W_out`)을 바꾸지 마세요.**
3. **"제공 코드 (수정 금지)"** 셀과 **고정된 하이퍼파라미터**(`EMB_DIM`, `EPOCHS`, `BATCH_SIZE`, `LR`, `SEED`)는 수정하지 마세요.
4. 디버깅용 셀은 자유롭게 추가해도 되지만, **제출 전 `런타임 → 모두 실행`으로 처음부터 끝까지 에러 없이 실행되는지 확인**하세요.
5. 사용 가능한 패키지: `numpy`, `pandas`, `torch`, `tqdm`, 파이썬 표준 라이브러리. (그 외 패키지 설치·사용 금지, `os`·`subprocess` 등으로 파일/시스템을 조작하는 코드 금지)
6. **Colab 사용법**: 이 노트북과 `MovieLens100K` 폴더를 Google Drive 의 **같은 폴더**에 올리고, 노트북을 Colab 으로 열어 첫 셀부터 실행하세요. 첫 셀이 Drive 를 마운트하고 그 폴더를 자동으로 찾아 이동합니다. 데이터 경로 `./MovieLens100K/` 는 그대로 두세요. (채점 시 첫 셀은 자동으로 건너뜁니다)
7. 제출 파일: **실행 결과가 남아있는** `기초추천시스템과제_학번_이름.ipynb` 1개.
8. 예상 실행 시간: Colab CPU 기준 약 3–5분 (학습 3 epoch 포함), GPU 사용 시 더 빠름.

### 배점 (총 100점)

| 문항 | 내용 | 배점 | 자동 채점 기준 |
|---|---|---|---|
| 1-1 | `filter_data` | 10 | 실제 데이터 + 숨김 데이터에서 결과가 정답과 일치 |
| 1-2 | `split_train_test` | 10 | 실제 데이터 + 숨김 데이터에서 train/test 가 정답과 일치 |
| 1-3 | `make_pairs` | 10 | 실제 데이터 + 숨김 데이터에서 pair 집합이 정답과 일치 |
| 2-1 | `MovieNet` | 10 | `W_in`/`W_out` 구조·출력 크기, 출력이 softmax 가 아닌 logit 인지 |
| 2-2 | `train_one_epoch` | 15 | 숨김 소형 데이터에서 학습 동작 + 학습된 `model` 의 loss < **6.08** |
| 2-3 | `get_movie_embeddings` | 15 | 반환값이 `W_in` 가중치로 만든 (num_items, EMB_DIM) 임베딩과 일치 |
| 3-1 | `get_user_embedding` | 10 | 실제 데이터 + 숨김 데이터에서 정답과 일치 |
| 3-2 | `recommend` | 10 | 실제 데이터 + 숨김 데이터에서 Top-k 영화·순서·컬럼이 정답과 일치 |
| 3-3 | 추천 성능 | 10 | 학습된 임베딩의 Hit@10 ≥ **0.30** (10점), ≥ 0.20 (5점) |

> 각 문항의 테스트가 두 가지(실제 데이터 / 숨김 데이터)인 경우, 하나만 통과하면 배점의 절반을 받습니다.
> 한 문항이 틀려도 다음 문항은 **정답 입력**으로 따로 채점하므로, 앞 문항 때문에 뒤 문항 점수가 깎이지 않습니다.

In [ ]:
# Google Drive 마운트 & 작업 폴더 이동 (제공 코드, 수정 금지) — 채점 시 자동으로 건너뜀
# 노트북과 MovieLens100K 폴더를 Google Drive 의 같은 폴더에 올려두면, 그 폴더를 자동으로 찾아 이동합니다.
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import glob, os
    drive.mount('/content/drive')
    found = []
    for depth in range(5):  # 내 드라이브 아래 5단계 폴더까지 검색
        found = glob.glob('/content/drive/MyDrive/' + '*/' * depth + 'MovieLens100K/ratings.csv')
        if found:
            break
    assert found, "Google Drive 에서 MovieLens100K 폴더를 찾지 못했습니다. 노트북과 같은 폴더에 MovieLens100K 폴더를 올렸는지 확인하세요."
    if len(found) > 1:
        print('MovieLens100K 폴더가 여러 개 있어 첫 번째를 사용합니다:', found)
    os.chdir(os.path.dirname(os.path.dirname(found[0])))
    print('작업 폴더:', os.getcwd())

# Import Modules & Setting (제공 코드, 수정 금지)

In [ ]:
import random
import warnings
from itertools import permutations  # pair 생성용

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm

warnings.filterwarnings('ignore')

SEED = 42
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

# Data Loading (제공 코드, 수정 금지)

In [ ]:
DATA_DIR = './MovieLens100K/'
df_ratings = pd.read_csv(DATA_DIR + 'ratings.csv')  # userId, movieId, rating, timestamp
df_movies = pd.read_csv(DATA_DIR + 'movies.csv')    # movieId, title, genres
print('ratings:', df_ratings.shape, '| movies:', df_movies.shape)
df_ratings.head()

# Problem 1. Preprocessing (30 points)

### 1-1. `filter_data` (10점)

평가가 너무 적은 영화·사용자는 임베딩을 제대로 학습하기 어렵기 때문에 제거합니다. **아래 순서를 지켜주세요.**

1. 평가 수가 `min_movie_ratings`(=50) **미만**인 **영화**의 평가를 모두 제거
2. 1번을 적용한 **이후** 남은 평가 수가 `min_user_ratings`(=10) **미만**인 **사용자**의 평가를 모두 제거
3. `reset_index(drop=True)` 로 index 를 0부터 다시 매겨서 반환 (컬럼은 그대로 유지)

> 힌트: `df.groupby('movieId')['userId'].transform('size')` 는 각 행에 "그 영화의 평가 수"를 붙여줍니다.

In [ ]:
def filter_data(df_ratings, min_movie_ratings=50, min_user_ratings=10):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# 자기 점검 (제공 코드, 수정 금지)
df = filter_data(df_ratings)
print(f"평가 {len(df):,}개 | 영화 {df['movieId'].nunique()}개 | 사용자 {df['userId'].nunique()}명")
print("참고 정답: 평가 41,084개 | 영화 450개 | 사용자 562명")

### 영화 index 부여 (제공 코드, 수정 금지)

신경망의 one-hot 입력을 만들려면 영화마다 `0 ~ num_items-1` 의 index 가 필요합니다. 아래 코드가 `df` 에 `item` 컬럼을 추가합니다.
- `item2movie[i]` : item index `i` 에 해당하는 원래 `movieId`
- `movie2item[m]` : `movieId` `m` 에 해당하는 item index

In [ ]:
def build_item_index(df):
    item2movie = np.sort(df['movieId'].unique())
    movie2item = {m: i for i, m in enumerate(item2movie)}
    df = df.copy()
    df['item'] = df['movieId'].map(movie2item)
    return df, item2movie, movie2item

df, item2movie, movie2item = build_item_index(df)
num_items = len(item2movie)
print('num_items:', num_items)
df.head()

### 1-2. `split_train_test` (10점) — slide 58

사용자마다 본 영화 중 **가장 최근에 본 `n_test`(=5)편을 test**, 나머지를 train 으로 나눕니다.

- "최근"의 기준: `timestamp` 가 큰 것. `timestamp` 가 같으면 `movieId` 가 큰 것을 더 최근으로 봅니다.
  - 즉, `['userId', 'timestamp', 'movieId']` 로 오름차순 정렬한 뒤 사용자별 **마지막 5개**가 test 입니다.
- 반환: `(train_df, test_df)` — 두 DataFrame 모두 입력 `df` 와 같은 컬럼을 가져야 합니다.

> 힌트: `groupby('userId').tail(n)` 은 그룹별 마지막 n개 행을 돌려줍니다. `df.drop(index)` 로 나머지를 구할 수 있습니다.

In [ ]:
def split_train_test(df, n_test=5):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# 자기 점검 (제공 코드, 수정 금지)
train_df, test_df = split_train_test(df)
print(f"train {len(train_df):,}개 | test {len(test_df):,}개")
print("참고 정답: train 38,274개 | test 2,810개")

### 1-3. `make_pairs` (10점) — slide 59

**train_df** 에서 사용자마다 본 영화(`item` index) 목록에 대해 **permutation** 으로 순서가 있는 pair 를 만듭니다.
- 예: 한 사용자가 영화 `[a, b, c]` 를 봤다면 → `(a,b), (a,c), (b,a), (b,c), (c,a), (c,b)` (6개)
- 자기 자신과의 pair `(a,a)` 는 만들지 않습니다.
- 반환: 모든 사용자의 pair 를 이어붙인 `np.ndarray`, shape `(전체 pair 수, 2)`, dtype `int64`. 각 행은 `[movie1 의 item, movie2 의 item]`.
- pair 의 순서는 상관없습니다. (학습 시 `DataLoader(shuffle=True)` 가 섞어줍니다)

> 힌트: `list(permutations(items, 2))`. 사용자별로 `np.array(...)` 로 바꾼 뒤 마지막에 `np.concatenate` 하면 빠릅니다.

In [ ]:
def make_pairs(train_df):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# 자기 점검 (제공 코드, 수정 금지)
pairs = make_pairs(train_df)
print('pairs shape:', pairs.shape, '| dtype:', pairs.dtype)
print("참고 정답: pairs shape (5391706, 2)")

# Problem 2. Learning Movie Embeddings (40 points) — slide 60

$(m_1, m_2)$ pair 로 **multi-class classification** (클래스 = 영화) 을 학습합니다.

```
one-hot(m1)  --W_in-->  embedding (EMB_DIM)  --W_out-->  logits (num_items)  --softmax + CE-->  정답 m2
 (num_items)            ← 이 층이 영화 임베딩
```

- **Input**: $m_1$ 의 one-hot 벡터 (크기 `num_items`)
- **Embedding layer**: `W_in = nn.Linear(num_items, EMB_DIM, bias=False)`
- **Output layer**: `W_out = nn.Linear(EMB_DIM, num_items)` → 각 영화에 대한 점수(logit)
- **Loss**: `nn.CrossEntropyLoss()` 와 정답 $m_2$ 의 item index

> ⚠️ **주의**: `nn.CrossEntropyLoss` 는 **내부에서 softmax 를 계산**합니다. 모델의 `forward` 에서 softmax 를 또 적용하면 softmax 가 두 번 걸려 **학습이 거의 되지 않습니다** (loss 가 $\ln(\text{num\_items}) \approx 6.11$ 근처에서 멈춤). `forward` 는 softmax 전의 **logit** 을 반환하세요.
>
> 참고: 정답을 one-hot 벡터 대신 index 로 주는 것은 one-hot 정답과의 cross-entropy 와 수학적으로 같습니다.

### 2-1. `MovieNet` (10점)
- `__init__` 에서 **반드시 `self.W_in`, `self.W_out` 이라는 이름**으로 위의 두 층을 만드세요.
- `forward(x)`: `x` 는 shape `(B, num_items)` 인 one-hot 행렬, 반환은 shape `(B, num_items)` 인 logit.

In [ ]:
class MovieNet(nn.Module):
    def __init__(self, num_items, emb_dim):
        super().__init__()
        # YOUR CODE HERE
        raise NotImplementedError()

    def forward(self, x):
        # YOUR CODE HERE
        raise NotImplementedError()

### 하이퍼파라미터 & DataLoader (제공 코드, 수정 금지)

In [ ]:
EMB_DIM = 32
EPOCHS = 3
BATCH_SIZE = 1024
LR = 1e-3

pair_tensor = torch.from_numpy(pairs)
train_loader = DataLoader(
    TensorDataset(pair_tensor[:, 0], pair_tensor[:, 1]),  # (movie1 index, movie2 index)
    batch_size=BATCH_SIZE, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)
print('batches per epoch:', len(train_loader))

### 2-2. `train_one_epoch` (15점)

한 epoch 동안 학습하고 **평균 train loss** 를 반환하는 함수입니다. 입력 one-hot 변환과 loss 누적은 이미 작성되어 있습니다.
`# YOUR CODE HERE` 부분에 다음 4단계를 작성하세요.

1. `model` 에 `x` 를 넣어 `logits` 계산
2. `criterion(logits, y)` 로 `loss` 계산
3. `optimizer.zero_grad()` → `loss.backward()`
4. `optimizer.step()`

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, num_items):
    model.train()
    total_loss, total_n = 0.0, 0
    for x_idx, y_idx in tqdm(loader, mininterval=10):
        x = F.one_hot(x_idx, num_classes=num_items).float().to(device)  # 입력: movie1 의 one-hot
        y = y_idx.to(device)                                             # 정답: movie2 의 index

        # YOUR CODE HERE
        raise NotImplementedError()

        total_loss += loss.item() * len(y_idx)
        total_n += len(y_idx)
    return total_loss / total_n

In [ ]:
# 학습 실행 (제공 코드, 수정 금지)
set_seed()
model = MovieNet(num_items, EMB_DIM).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

loss_history = []
for epoch in range(EPOCHS):
    epoch_loss = train_one_epoch(model, train_loader, optimizer, criterion, num_items)
    loss_history.append(epoch_loss)
    print(f"Epoch {epoch + 1}/{EPOCHS} | train loss {epoch_loss:.4f}")

print(f"\n참고: 학습 전(무작위 예측) loss ≈ ln(num_items) = {np.log(num_items):.4f}")
print("참고 정답: 3 epoch 후 train loss ≈ 6.02 (6.08 미만이면 통과)")

### 2-3. `get_movie_embeddings` (15점)

학습이 끝난 `W_in` 의 가중치가 곧 영화 임베딩입니다. one-hot 벡터 $e_i$ 에 `W_in` 을 곱하면 가중치 행렬의 $i$ 번째 열이 나오므로, **$i$ 번째 열 = item $i$ 의 임베딩**입니다.

- `nn.Linear(num_items, EMB_DIM)` 의 `.weight` shape 은 `(EMB_DIM, num_items)` 입니다. (출력 크기가 앞!)
- 반환: shape `(num_items, EMB_DIM)` 인 `torch.Tensor` — **행 $i$ 가 item $i$ 의 임베딩**이 되도록 만드세요.
- 반환 텐서는 gradient 추적을 끊고(`.detach()`) CPU 로 옮겨주세요(`.cpu()`).

In [ ]:
def get_movie_embeddings(model):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# 자기 점검 (제공 코드, 수정 금지)
movie_emb = get_movie_embeddings(model)
print('movie_emb shape:', tuple(movie_emb.shape), '| 참고 정답: (450, 32)')

# Problem 3. Recommendation (30 points) — slide 61

### 3-1. `get_user_embedding` (10점)

사용자가 **train 에서 본 영화들의 임베딩을 평균**(element-wise mean)하여 사용자 임베딩 1개를 만듭니다.
- 반환: shape `(EMB_DIM,)` 인 `torch.Tensor`
- `train_df` 의 `item` 컬럼이 `movie_emb` 의 행 번호입니다.

In [ ]:
def get_user_embedding(user_id, train_df, movie_emb):
    # YOUR CODE HERE
    raise NotImplementedError()

### 3-2. `recommend` (10점)

1. `get_user_embedding` 으로 사용자 임베딩을 구합니다.
2. 사용자 임베딩과 **모든 영화 임베딩 사이의 cosine similarity** 를 계산합니다. (힌트: `F.cosine_similarity(movie_emb, user_emb.unsqueeze(0), dim=1)`)
3. 사용자가 **train 에서 이미 본 영화는 추천에서 제외**합니다.
4. 유사도가 가장 높은 `k` 개 영화를 **유사도 내림차순**으로 반환합니다.

- 반환: 길이 `k` 인 `pd.DataFrame`, 컬럼은 정확히 **`['movieId', 'title', 'genres', 'similarity']`** 순서.
  - `movieId` 는 item index 가 아니라 **원래 movieId** 입니다. (`item2movie` 사용)

In [ ]:
def recommend(user_id, train_df, movie_emb, df_movies, item2movie, k=10):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
# 예시: userId = 1 에게 추천 (제공 코드, 수정 금지)
print("[userId=1 이 실제로 나중에 본 영화 (test)]")
print(test_df[test_df['userId'] == 1].merge(df_movies, on='movieId')[['movieId', 'title', 'genres']].to_string(index=False))
print("\n[userId=1 에게 추천한 Top-10]")
recommend(1, train_df, movie_emb, df_movies, item2movie, k=10)

### 3-3. 추천 성능 평가 (10점, 제공 코드 · 수정 금지)

**Hit@10**: 추천한 10편 중 test 영화(사용자가 실제로 나중에 본 5편)가 **1편 이상** 포함된 사용자의 비율.
위 문항들을 올바르게 구현했다면 별도 코드 작성 없이 이 점수가 기준을 넘습니다.

| 방법 | Hit@10 (참고) |
|---|---|
| 무작위 추천 | ≈ 0.11 |
| 인기순 추천 (train 에서 많이 본 영화) | ≈ 0.26 |
| **학습된 임베딩 (이 과제)** | **≈ 0.39–0.41** |

In [ ]:
def hit_at_k(train_df, test_df, movie_emb, df_movies, item2movie, k=10):
    test_sets = test_df.groupby('userId')['movieId'].apply(set)
    hits = []
    for user_id, test_movies in tqdm(test_sets.items(), total=len(test_sets), mininterval=10):
        rec = recommend(user_id, train_df, movie_emb, df_movies, item2movie, k)
        hits.append(len(set(rec['movieId']) & test_movies) > 0)
    return float(np.mean(hits))

hit10 = hit_at_k(train_df, test_df, movie_emb, df_movies, item2movie, k=10)
print(f"Hit@10 = {hit10:.4f}  (기준: 0.30 이상 10점, 0.20 이상 5점)")

# 제출 전 최종 확인 (제공 코드, 수정 금지)

`런타임 → 모두 실행` 후 아래 셀의 출력을 확인하고 제출하세요.

In [ ]:
print("=" * 50)
print(f"[1-1] 평가 수            : {len(df):,} (정답 41,084)")
print(f"[1-2] train / test       : {len(train_df):,} / {len(test_df):,} (정답 38,274 / 2,810)")
print(f"[1-3] pairs              : {len(pairs):,} (정답 5,391,706)")
print(f"[2-2] 최종 train loss    : {loss_history[-1]:.4f} (기준 < 6.08)")
print(f"[2-3] movie_emb shape    : {tuple(movie_emb.shape)} (정답 (450, 32))")
print(f"[3-3] Hit@10             : {hit10:.4f} (기준 ≥ 0.30)")
print("=" * 50)